# 👂 OPERAÇÃO VOZ DO CLIENTE · FASE 2
### Simulado de NLP 1 · Trilha de Linguagem Natural · o acervo da Fase 1, com modelo de linguagem

---

> **"O modelo da Fase 1 tropeçava em 'recomendissimo' e 'prodto'. Disseram que agora a
> máquina lê como gente. Quero ver."**
>
> — Vera Nogueira, Ouvidoria

## 📨 A Ouvidoria voltou

> É **o mesmo acervo** da Fase 1: mais de cem mil avaliações reais de clientes, escritas
> às pressas, com abreviação, erro de digitação e gíria. A tarefa é a mesma: **satisfeito**
> ou **insatisfeito** — e o insatisfeito continua sendo o caro de deixar passar.
>
> A diferença é o modelo. Em vez de contar palavras de uma lista, vocês vão ajustar um
> modelo que **já lê português**. E eu quero saber, com números, se ele lê melhor que o
> TF-IDF de vocês — e onde ele ainda tropeça.
>
> — Vera Nogueira

---

*Mesmos dados oficiais da Operação Voz do Cliente. O teste oficial não tem gabarito
público; 900 avaliações rotuladas viraram o teste oculto deste simulado.*

## 🎯 A sua missão


Ajustar um **modelo de linguagem pré-treinado** — **DistilBERT multilíngue**, **BERTimbau**
ou **XLM-RoBERTa** — para decidir se o cliente está satisfeito, e comparar com a marca da
Fase 1 (F1-macro ≈ 0,95 com TF-IDF + SVM linear). Superar essa marca não é garantido: medir
isso também é resultado.


### O que muda da Fase 1 para cá

| Na Fase 1 | Na Fase 2 |
|:---|:---|
| TF-IDF: contar palavras de uma lista fechada | **Pedaços de palavra**: nenhuma palavra é jogada fora |
| Palavra nunca vista sumia; erro de digitação virava palavra desconhecida | `prodto` vira `prod ##to`: o modelo entende as partes |
| Limpar o texto (minúsculas, acentos) ajudava | **Não limpem**: os modelos usam maiúscula e acento |
| O modelo nascia vazio | O modelo chega **sabendo português**; o ajuste fino ensina só a tarefa |

### 🗺️ O mapa da operação

| Bloco | O que vocês fazem | Como |
|:---|:---|:---|
| **1 · EXPLORAR** | O piso, os três tradutores e o limite de corte | ✏️ **1 lacuna** |
| **2 · CONFIGURAR** | Escolher o modelo e girar os quatro botões | Só arrastando |
| **3 · MODELAR** | Células A (preparo), B (treino) e C (avaliação) | ✏️ **9 lacunas** + Zona Livre |
| **4 · AVALIAR** | Curvas, matriz com nome próprio, F1 por grupo de texto | Só clicando |

> ⚠️ **Liguem a GPU.** `Ambiente de execução` → `Alterar o tipo de ambiente de execução` →
> **T4 GPU**. Ajuste fino de modelo de linguagem em CPU leva horas; na T4, minutos. A
> primeira célula avisa se a GPU não estiver ligada.
>
> 📎 **Simulado de preparação.** Formato dos cadernos da Fase 2, com dados públicos. O
> gabarito do teste fica com o corretor (`simulados/corrigir.py`), nunca dentro do caderno.

In [ ]:
#@title 🛠️ Preparação: abrindo o acervo { display-mode: "form" }
URL_TREINO = ""
URL_TESTE = ""
SEMENTE = 42
import pathlib, warnings, re
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, recall_score, precision_score,
                             confusion_matrix, classification_report)
import torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)
from transformers.utils import logging as _hf_logging
_hf_logging.set_verbosity_error()

NOME_CLASSE = {0: 'INSATISFEITO', 1: 'SATISFEITO'}
CLASSE_CRITICA = 0   # a que custa caro deixar passar

def _carregar(url, nome_local):
    """Usa a cópia ao lado do caderno; senão tenta o endereço; senão pede o envio."""
    if pathlib.Path(nome_local).exists():
        return pd.read_csv(nome_local)
    if url:
        print(f"Baixando {nome_local}...")
        try:
            return pd.read_csv(url)
        except Exception as erro:
            print(f"  não deu certo ({erro.__class__.__name__}).")
    from google.colab import files
    print(f"  envie o arquivo {nome_local}:")
    files.upload()
    return pd.read_csv(nome_local)

dados = _carregar(URL_TREINO, "treino.csv")
_teste = _carregar(URL_TESTE, "teste.csv")
mensagens_teste = _teste["mensagem"].tolist()
IDS_TESTE = _teste["id_mensagem"].to_numpy()
N_TESTE = len(mensagens_teste)
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"

_p0 = (dados.rotulo == 0).mean() * 100
print("Ambiente carregado.\n")
print(f"  Acervo público (rotulado) ......... {len(dados):>6} textos")
print(f"     {NOME_CLASSE[0].lower():<14} (0) ......... {(dados.rotulo==0).sum():>6}   {_p0:4.1f}%")
print(f"     {NOME_CLASSE[1].lower():<14} (1) ......... {(dados.rotulo==1).sum():>6}   {100-_p0:4.1f}%")
print(f"  Conjunto de teste (sem rótulo) .... {N_TESTE:>6} textos  [tamanho fixo]")
print(f"  Dispositivo ....................... {DISPOSITIVO}")
if DISPOSITIVO == "cpu":
    print("\n[!] Sem GPU. Liguem a T4 antes do Bloco 3: em CPU o treino leva horas.")

---
# 🔍 BLOCO 1 · EXPLORAR
## *Como o texto é partido, e quanto dele cabe*


> **Vera:** *"Olhem o que acontece com 'recomendissimo' em cada tradutor."*

In [ ]:
# ✏️ SUA VEZ · O piso
y_publico = dados["rotulo"].to_numpy()
chute = np.full_like(y_publico, np.bincount(y_publico).argmax())
# LACUNA 1 ▸ que média do F1 é a métrica da fase?
piso = f1_score(y_publico, chute, average=____)
print(f"Responder '{NOME_CLASSE[int(chute[0])]}' para tudo: acurácia "
      f"{accuracy_score(y_publico, chute)*100:.1f}% · F1-macro {piso*100:.1f}%")
print("Esse F1-macro é o piso: o modelo de vocês precisa passar longe dele.")

In [ ]:
#@title 📬 Leia textos do acervo { display-mode: "form" }
tipo = "insatisfeito"  #@param ["insatisfeito", "satisfeito"]
quantidade = 5  #@param {type:"slider", min:2, max:10, step:1}
semente_amostra = 7  #@param {type:"integer"}
alvo = 0 if tipo == NOME_CLASSE[0].lower() else 1
for k, t in enumerate(dados[dados.rotulo == alvo].mensagem.sample(quantidade, random_state=semente_amostra), 1):
    print(f"[{k}] {t[:300]}{' [...]' if len(t) > 300 else ''}\n")

In [ ]:
#@title 🧩 Três tradutores, uma frase { display-mode: "form" }
#@markdown Troquem a frase: uma comum, uma com erro de digitação, uma com gíria.
frase = "chegou tudo certinho, top demais, recomendo mto"  #@param {type:"string"}
NOMES_MODELOS = {"DistilBERT multilíngue": "distilbert-base-multilingual-cased",
                 "BERTimbau": "neuralmind/bert-base-portuguese-cased",
                 "XLM-RoBERTa": "xlm-roberta-base"}
if "_tradutores" not in dir():
    _tradutores = {k: AutoTokenizer.from_pretrained(v) for k, v in NOMES_MODELOS.items()}
for k, tr_ in _tradutores.items():
    pedacos = tr_.tokenize(frase)
    print(f"{k:<24} {len(pedacos):>3} pedaços · vocabulário de {tr_.vocab_size//1000} mil")
    print("   " + " · ".join(pedacos) + "\n")
print("Nenhum dos três joga palavra fora: o que não conhecem, partem em pedaços conhecidos.")
print("O especialista em português conhece mais palavras nossas inteiras: parte menos.")

In [ ]:
#@title 📏 Quanto cabe no limite? { display-mode: "form" }
#@markdown Mova o limite e veja quantos textos perdem o fim no corte.
limite_em_teste = 128  #@param {type:"slider", min:16, max:512, step:16}
tradutor_para_medir = "BERTimbau"  #@param ["DistilBERT multilíngue", "BERTimbau", "XLM-RoBERTa"]
_amostra = dados.mensagem.sample(min(2000, len(dados)), random_state=SEMENTE).tolist()
_tam = np.array([len(x) for x in _tradutores[tradutor_para_medir](_amostra)["input_ids"]])
cortados = (_tam > limite_em_teste).mean()
fig, ax = plt.subplots(figsize=(8.5, 3.8))
ax.hist(_tam.clip(max=600), bins=60, color="#00B8D4")
ax.axvline(limite_em_teste, color="#E6177A", lw=2, label=f"limite {limite_em_teste}")
ax.set_xlabel("pedaços por texto"); ax.set_ylabel("textos"); ax.legend()
ax.set_title(f"Tamanho dos textos em pedaços ({tradutor_para_medir})", fontweight="bold")
ax.spines[["top", "right"]].set_visible(False); plt.tight_layout(); plt.show()
print(f"Mediana: {np.median(_tam):.0f} pedaços · 90% dos textos têm até {np.percentile(_tam, 90):.0f}.")
print(f"Com limite {limite_em_teste}, {cortados*100:.0f}% dos textos perdem o fim no corte.")
print("O custo de cada passo cresce com o QUADRADO do limite. O número que justifica a")
print("escolha de vocês é este percentual: anotem e levem para o Bloco 2.")

---
# ⚙️ BLOCO 2 · CONFIGURAR
## *Trocar de modelo é trocar uma palavra*

| Conjunto | Tamanho | Vocês têm os rótulos? | Para que serve |
|:---|:---|:---|:---|
| **Público** | 109.082 textos | Sim | Treinar e validar |
| **Teste** | **900 (fixo)** | Não | Comparação final (`simulados/corrigir.py`) |

### 📗 Um especialista e dois generalistas

| | 🪶 DistilBERT multilíngue | 🇧🇷 BERTimbau | 🌍 XLM-RoBERTa |
|:---|:---|:---|:---|
| **Nome** | `distilbert-base-multilingual-cased` | `neuralmind/bert-base-portuguese-cased` | `xlm-roberta-base` |
| **O que é** | Um modelo grande encolhido para metade das camadas | Treinado **só em português brasileiro** | Generalista treinado em 2,5 TB de texto em cem idiomas |
| **Camadas · vocabulário** | 6 · 119 mil pedaços | 12 · 30 mil pedaços | 12 · 250 mil pedaços |
| **Taxa** | `3e-5` a `5e-5` | `2e-5` | `1e-5` a `2e-5` |
| **Lote · épocas** | 32 ou 64 · 3 a 4 | 32 · 3 | 16 ou 32 · 3 |
| **A pegadinha** | Fala 104 idiomas e nenhum muito bem: perde para o especialista, e medir essa perda é parte do trabalho | Usa maiúsculas e acentos: **não passem o texto para minúsculas** como na Fase 1 | Não é mais fundo que o BERTimbau; é mais pesado por causa do vocabulário. O mais sensível à taxa e ao aquecimento |
| **O que esperar** | ~2× mais rápido; bom para as primeiras voltas | Tende a dar o melhor número pelo menor risco | Pode ganhar, pode divergir |

### A regra que não pode ser quebrada

**O tradutor e o modelo saem do mesmo nome.** O tradutor parte a frase em pedaços e numera cada
pedaço pela tabela dele; o modelo lê esses números na tabela dele. Com nomes diferentes, cada
número cai numa linha errada e o modelo lê uma frase que ninguém escreveu. **Não dá erro**: o
código roda, a barra anda, e o resultado despenca.

### Quatro botões, e só quatro

| Botão | Faixa de trabalho | Fora dela |
|:---|:---|:---|
| **Taxa** | `1e-5` a `5e-5` | `1e-3` apaga o pré-treino em poucas centenas de passos |
| **Épocas** | 3 | o ajuste fino não ensina a língua, ensina a tarefa: **a quarta costuma estragar** |
| **Limite** | o medido no Bloco 1 (128 é o padrão) | curto: decapita o texto; longo: o custo cresce com o **quadrado** do limite |
| **Aquecimento** | 10% dos passos | sem ele a primeira época sofre (o XLM-R é o que mais sofre) |

> Os valores iniciais abaixo são **deliberadamente modestos** (e mais de um está fora da
> ficha). O acervo é enorme: `fracao_do_acervo = 0,2` (~22 mil avaliações) já cabe em poucos minutos de T4. Comecem assim e só aumentem se a curva pedir.

In [ ]:
#@title 🎛️ Parâmetros do experimento { display-mode: "form" }
modelo_escolhido = "DistilBERT multilíngue"  #@param ["DistilBERT multilíngue", "BERTimbau", "XLM-RoBERTa"]
tamanho_da_validacao = 10  #@param {type:"slider", min:5, max:30, step:5}
fracao_do_acervo = 0.2  #@param {type:"slider", min:0.005, max:1.0, step:0.005}
taxa = 5e-5  #@param [1e-3, 1e-4, 5e-5, 3e-5, 2e-5, 1e-5] {type:"raw"}
epocas = 4  #@param {type:"slider", min:1, max:8, step:1}
limite = 64  #@param [32, 64, 128, 256, 512] {type:"raw"}
aquecimento = 0  #@param {type:"slider", min:0, max:30, step:5}
lote = 32  #@param [8, 16, 32, 64] {type:"raw"}

FICHA = {"DistilBERT multilíngue": (3e-5, 5e-5), "BERTimbau": (2e-5, 2e-5), "XLM-RoBERTa": (1e-5, 2e-5)}
print("PARÂMETROS DEFINIDOS")
print(f"   Modelo ......... {modelo_escolhido}")
print(f"   Taxa ........... {taxa:g}   (ficha: {FICHA[modelo_escolhido][0]:g} a {FICHA[modelo_escolhido][1]:g})")
print(f"   Épocas ......... {epocas}   |   limite {limite}   |   aquecimento {aquecimento}%   |   lote {lote}")
print(f"   Acervo ......... {fracao_do_acervo:.1%} ({int(len(dados)*fracao_do_acervo)} textos)")
if taxa > FICHA[modelo_escolhido][1] * 2:
    print("\n[!] Taxa acima da ficha. Num modelo que já sabe português, passo grande atropela.")
if epocas > 3:
    print("\n[!] Mais de três épocas: o ajuste fino ensina só a tarefa, e a quarta costuma estragar.")
if aquecimento == 0:
    print("\n[!] Sem aquecimento: a primeira época leva passos de tamanho cheio antes de se orientar.")

---
# ✏️ BLOCO 3 · MODELAR
## *As mesmas três células, pela terceira vez*

Dois erros desta trilha **não dão mensagem nenhuma**, e a verificação procura os dois:

1. **Tradutor e modelo de nomes diferentes.** Cada número de pedaço cai na linha errada da
   tabela do modelo. Antes de avaliar, o caderno compara os dois nomes.
2. **Limpar o texto como na Fase 1.** Passar para minúsculas destrói informação que estes
   modelos usam.

> ⚠️ Preencham todas as lacunas antes de executar.

In [ ]:
#@title 🧰 Peças prontas: o laço de treino e as medidas { display-mode: "form" }
import time, copy
from torch.optim import AdamW
from torch.utils.data import TensorDataset, DataLoader
torch.manual_seed(SEMENTE)


def carregador(entradas, rotulos, lote, embaralhar):
    return DataLoader(TensorDataset(entradas["input_ids"], entradas["attention_mask"],
                                    torch.as_tensor(np.asarray(rotulos))),
                      batch_size=lote, shuffle=embaralhar)


@torch.no_grad()
def prever(modelo, dados):
    """Probabilidade de cada classe. NÃO liga o modo de avaliação sozinho."""
    probs = []
    for ids, mascara, _ in dados:
        saida = modelo(input_ids=ids.to(DISPOSITIVO), attention_mask=mascara.to(DISPOSITIVO))
        probs.append(torch.softmax(saida.logits, dim=1).cpu())
    return torch.cat(probs).numpy()


def treinar(modelo, dados_tr, dados_va, otimizador, agendador, perda, epocas,
            ponto_de_verificacao):
    historico = {"treino": [], "validacao": [], "passos": []}
    melhor_perda, melhor_epoca, melhor_pesos = float("inf"), 0, None
    for epoca in range(1, epocas + 1):
        modelo.train()
        soma, n = 0.0, 0
        for k, (ids, mascara, resposta) in enumerate(dados_tr):
            ids, mascara, resposta = ids.to(DISPOSITIVO), mascara.to(DISPOSITIVO), resposta.to(DISPOSITIVO)
            otimizador.zero_grad()                                          # 1
            saida = modelo(input_ids=ids, attention_mask=mascara).logits    # 2
            erro = perda(saida, resposta)                                   # 3
            erro.backward()                                                 # 4
            torch.nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
            otimizador.step()                                               # 5
            agendador.step()        # aqui o passo muda a cada LOTE (aquecimento + decaimento)
            soma += erro.item() * len(resposta); n += len(resposta)
            historico["passos"].append(erro.item())
            if k % 50 == 0:
                print(f"   época {epoca} · lote {k:>4}/{len(dados_tr)} · perda {erro.item():.3f}", flush=True)
        modelo.eval()
        with torch.no_grad():
            s, m = 0.0, 0
            for ids, mascara, resposta in dados_va:
                ids, mascara, resposta = ids.to(DISPOSITIVO), mascara.to(DISPOSITIVO), resposta.to(DISPOSITIVO)
                s += perda(modelo(input_ids=ids, attention_mask=mascara).logits, resposta).item() * len(resposta)
                m += len(resposta)
        historico["treino"].append(soma / n); historico["validacao"].append(s / m)
        print(f"   ÉPOCA {epoca}: perda treino {soma/n:.3f} · validação {s/m:.3f}", flush=True)
        if s / m < melhor_perda:
            melhor_perda, melhor_epoca = s / m, epoca
            if ponto_de_verificacao:
                melhor_pesos = {k: v.detach().cpu().clone() for k, v in modelo.state_dict().items()}
    if ponto_de_verificacao and melhor_pesos is not None:
        modelo.load_state_dict(melhor_pesos)
    historico["melhor_epoca"] = melhor_epoca
    return historico


print("Peças prontas.")

In [ ]:
# ═══ CÉLULA A · preparo ═══
NOME = NOMES_MODELOS[modelo_escolhido]
# LACUNA 2 ▸ o tradutor sai do MESMO nome que os pesos.
tradutor = AutoTokenizer.from_pretrained(____)

_sub = dados.sample(frac=fracao_do_acervo, random_state=SEMENTE) if fracao_do_acervo < 1 else dados
# LACUNA 3 ▸ o texto que entra no tradutor. NÃO limpem como na Fase 1 (sem .lower()).
textos = ____
textos = textos.loc[_sub.index].tolist()
y = _sub["rotulo"].to_numpy()

# LACUNA 4 ▸ separar mantendo a proporção das classes
X_tr, X_va, y_tr, y_va = train_test_split(textos, y, test_size=tamanho_da_validacao/100,
                                          random_state=SEMENTE, stratify=____)
# LACUNA 5 ▸ quantos pedaços cabem? (o limite medido no Bloco 1)
tokenizar = lambda xs: tradutor(xs, truncation=True, padding="max_length",
                                max_length=____, return_tensors="pt")
entradas_tr, entradas_va = tokenizar(X_tr), tokenizar(X_va)
dados_tr = carregador(entradas_tr, y_tr, lote, embaralhar=True)
dados_va = carregador(entradas_va, y_va, 64, embaralhar=False)
print(f"Treino: {len(X_tr)} · validação: {len(X_va)} · entrada: {entradas_tr['input_ids'].shape}")

In [ ]:
# ═══ CÉLULA B · treino ═══
# LACUNA 6 ▸ os pesos saem de qual nome? (o mesmo do tradutor — sempre)
modelo = AutoModelForSequenceClassification.from_pretrained(____, num_labels=2).to(DISPOSITIVO)

# ═══ ZONA LIVRE ═══  (vazia = correto e mediano; um ajuste por vez)
peso_da_classe_critica = 1.0   # 1.4 deu bons resultados na Fase 1 (TF-IDF)
decaimento_de_pesos = 0.01
# ══════════════════
perda = torch.nn.CrossEntropyLoss(weight=torch.tensor([peso_da_classe_critica, 1.0], device=DISPOSITIVO))

# LACUNA 7 ▸ quem move os pesos?
otimizador = ____(modelo.parameters(), lr=taxa, weight_decay=decaimento_de_pesos)
total = len(dados_tr) * epocas
# LACUNA 8 ▸ quantos passos de aquecimento? ('aquecimento' é uma porcentagem de 'total')
agendador = get_linear_schedule_with_warmup(otimizador, num_warmup_steps=____,
                                            num_training_steps=total)
# LACUNA 9 ▸ guardar a melhor época, e não a última?
ponto_de_verificacao = ____

inicio = time.time()
historico = treinar(modelo, dados_tr, dados_va, otimizador, agendador, perda, epocas,
                    ponto_de_verificacao)
tempo_de_treino = time.time() - inicio
print(f"\nTreino concluído em {tempo_de_treino/60:.1f} min · melhor época: {historico['melhor_epoca']}")

In [ ]:
# ═══ CÉLULA C · avaliação ═══
# LACUNA 10 ▸ antes de prever: desligar o dropout e o modo de treino.
modelo.____()
prob_va = prever(modelo, dados_va)
previsoes = prob_va.argmax(axis=1)
f1 = f1_score(y_va, previsoes, average="macro")
acuracia = accuracy_score(y_va, previsoes)
recall_critica = recall_score(y_va, previsoes, pos_label=CLASSE_CRITICA)
precisao_critica = precision_score(y_va, previsoes, pos_label=CLASSE_CRITICA, zero_division=0)
if "registro" not in dir():
    registro = []
registro.append({"modelo": modelo_escolhido, "taxa": taxa, "epocas": epocas, "limite": limite,
                 "aquecimento": aquecimento, "fracao": fracao_do_acervo,
                 "melhor_epoca": historico["melhor_epoca"], "f1_macro": round(f1, 4),
                 "recall_critica": round(recall_critica, 4), "minutos": round(tempo_de_treino/60, 1)})
_nc = NOME_CLASSE[CLASSE_CRITICA]
print("="*62)
print("             DESEMPENHO NO CONJUNTO DE VALIDAÇÃO")
print("="*62)
print(f"  Piso (chute preguiçoso) .................. {piso*100:5.1f}%")
print(f"  ⭐ F1-MACRO (métrica oficial) ............. {f1*100:5.1f}%")
print(f"  🔴 Recall de {_nc.lower():<14} (apoio) ....... {recall_critica*100:5.1f}%")
print(f"     Precisão de {_nc.lower():<14} ............. {precisao_critica*100:5.1f}%")
print("="*62)
a, s_, i_ = (0.95, 0.93, 0.9)
if   f1 >= a:  print("  Desempenho alto. Agora olhem o F1 por grupo de texto (Bloco 4).")
elif f1 >= s_: print("  Desempenho sólido. Há margem nos quatro botões e na Zona Livre.")
elif f1 >= i_: print("  Desempenho intermediário. Leiam as curvas antes de mexer.")
else:          print("  Desempenho baixo. Confiram a taxa e a regra do mesmo nome.")

In [ ]:
#@title 🔎 Verificação das lacunas (os erros que não dão erro) { display-mode: "form" }
_estava_em_treino = modelo.training
erros = []
try:
    if not np.isclose(piso, f1_score(y_publico, chute, average="macro")):
        erros.append("LACUNA 1: o piso não é o F1-macro do chute preguiçoso.")
except NameError:
    erros.append("LACUNA 1: a variável 'piso' não existe.")
_nome_tradutor = tradutor.name_or_path
_nome_modelo = modelo.config._name_or_path
if _nome_tradutor != _nome_modelo:
    erros.append(f"LACUNAS 2 e 6: TRADUTOR E MODELO DE NOMES DIFERENTES ('{_nome_tradutor}' × "
                 f"'{_nome_modelo}'). Cada pedaço foi numerado por uma tabela e lido em outra: o "
                 "modelo leu frases que ninguém escreveu, e nenhuma mensagem de erro apareceu.")
elif _nome_modelo != NOME:
    erros.append(f"LACUNAS 2 e 6: os dois saíram de '{_nome_modelo}', não do modelo escolhido ({NOME}).")
_originais = dados.loc[_sub.index, "mensagem"].tolist()
if textos != _originais:
    erros.append("LACUNA 3: o texto foi alterado antes do tradutor (minúsculas? limpeza?). "
                 "Estes modelos usam maiúscula e acento: entreguem o texto cru.")
_esp = np.bincount(y, minlength=2) * len(y_tr) / len(y)
if np.abs(np.bincount(y_tr, minlength=2) - _esp).max() > 1.5:
    erros.append("LACUNA 4: o treino não manteve a proporção das classes (stratify).")
if entradas_tr["input_ids"].shape[1] != limite:
    erros.append(f"LACUNA 5: a entrada tem {entradas_tr['input_ids'].shape[1]} pedaços, não o limite {limite}.")
if type(otimizador).__name__ != "AdamW":
    erros.append(f"LACUNA 7: esperado AdamW, obtido {type(otimizador).__name__}.")
_f0 = agendador.lr_lambdas[0](0)
if aquecimento > 0 and _f0 >= 1:
    erros.append("LACUNA 8: o aquecimento está ligado no Bloco 2, mas o agendador começa com o passo cheio.")
if ponto_de_verificacao is not True:
    erros.append("LACUNA 9: sem ponto de verificação, a entrega é o modelo da última época.")
if _estava_em_treino:
    erros.append("LACUNA 10: a célula C previu com o modelo em modo de TREINO (dropout ligado).")
if erros:
    print("VERIFICAÇÃO REPROVADA. Pontos a revisar:\n")
    for e in erros: print("   [!] " + e + "\n")
    # O caderno para aqui: seguir com uma lacuna errada é deixar a nota mentir.
    raise RuntimeError("VERIFICAÇÃO REPROVADA: corrijam os pontos acima antes de seguir.")
else:
    print("VERIFICAÇÃO APROVADA\n")
    print(f"   Tradutor e modelo do mesmo nome ({NOME}), texto cru, limite {limite}, melhor época.")
modelo.eval()

---
# 📈 BLOCO 4 · AVALIAR
## *F1-macro decide, recall aparece*

| Formato das curvas | O que significa numa trilha de texto |
|:---|:---|
| O melhor ponto vem cedo (2ª ou 3ª época) | **Como deve ser**: o modelo já sabia português |
| Treino desce, validação sobe depois da 3ª | **Uma época a mais**: a entrega leva a melhor, não a última |
| A primeira época começa num degrau alto | **Sem aquecimento**: o defeito mais barato de corrigir |
| As duas serrilham e não descem | **Passo grande demais**: o pré-treino foi apagado |

In [ ]:
#@title 📉 As curvas { display-mode: "form" }
tr, va = np.array(historico["treino"]), np.array(historico["validacao"])
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
ep = np.arange(1, len(tr) + 1)
axes[0].plot(ep, tr, "-o", color="#3B5BDB", lw=2, label="treino")
axes[0].plot(ep, va, "--o", color="#E6177A", lw=2, label="validação")
axes[0].axvline(historico["melhor_epoca"], color="#2F9E44", ls=":", lw=2, label="melhor época")
axes[0].set_xlabel("épocas"); axes[0].set_ylabel("perda"); axes[0].legend()
axes[0].set_title("Por época", fontweight="bold")
_p = pd.Series(historico["passos"]).rolling(20, min_periods=1).mean()
axes[1].plot(_p, color="#3B5BDB"); axes[1].set_xlabel("lotes"); axes[1].set_ylabel("perda de treino (média móvel)")
axes[1].set_title("Lote a lote: o aquecimento aparece no começo", fontweight="bold")
for a_ in axes: a_.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()
if f1 < piso + .05:
    diag = "PASSO GRANDE DEMAIS (ou regra do nome quebrada): o modelo não sai do piso."
elif len(va) > historico["melhor_epoca"] and va[-1] > va.min() * 1.05:
    diag = "UMA ÉPOCA A MAIS: a validação virou depois da melhor época."
elif aquecimento == 0 and len(historico["passos"]) > 40 and np.mean(historico["passos"][:10]) > 1.2 * np.mean(historico["passos"][10:40]):
    diag = "SEM AQUECIMENTO: a perda começa num degrau alto. Liguem 10% de aquecimento."
else:
    diag = "COMO DEVE SER: o melhor ponto vem cedo e a validação assenta."
print(f"DIAGNÓSTICO: {diag}")

In [ ]:
#@title 🧩 Com duas classes, a matriz tem nome próprio { display-mode: "form" }
# Aqui o "positivo" é a classe crítica: é ela que se procura.
pos, neg = CLASSE_CRITICA, 1 - CLASSE_CRITICA
VP = int(((y_va == pos) & (previsoes == pos)).sum()); FN = int(((y_va == pos) & (previsoes == neg)).sum())
FP = int(((y_va == neg) & (previsoes == pos)).sum()); VN = int(((y_va == neg) & (previsoes == neg)).sum())
_np, _nn = NOME_CLASSE[pos].lower(), NOME_CLASSE[neg].lower()
print(f"                         o modelo disse {_np:<14} o modelo disse {_nn}")
print(f"   era {_np:<14}      VP = {VP:<5} (achou)          FN = {FN:<5} (passou batido)")
print(f"   era {_nn:<14}      FP = {FP:<5} (alarme falso)   VN = {VN:<5}")
print(f"\n   FALSO NEGATIVO: {FN} {_np}s que ninguém vai olhar. Ninguém procura esse cliente, e a próxima notícia dele é uma reclamação pública.")
print(f"   FALSO POSITIVO: {FP} alarmes falsos. Um contato desnecessário: custa tempo da ouvidoria, não um cliente perdido.")
print("\n   Qual dos dois custa mais caro? Não existe resposta técnica: depende do que o")
print("   sistema faz com a previsão — e quem decide isso é gente, não métrica.\n")
print(classification_report(y_va, previsoes, target_names=[NOME_CLASSE[0], NOME_CLASSE[1]], digits=3))

In [ ]:
#@title 👥 Com quem o modelo erra mais: F1-macro por grupo de texto { display-mode: "form" }
_txt = pd.Series(X_va)
_pal = _txt.str.split().str.len()
_ntok = np.array([len(x) for x in tradutor(list(X_va))["input_ids"]])
grupos = {"geral (a nota do ranking)": np.ones(len(X_va), bool),
          "textos curtos (até 20 palavras)": (_pal <= 20).to_numpy(),
          "textos longos (mais de 80 palavras)": (_pal > 80).to_numpy(),
          "com negação (não, nunca, nem)": _txt.str.contains(r"\b(?:n[ãa]o|nunca|nem)\b", case=False).to_numpy(),
          f"cortados pelo limite ({limite})": _ntok > limite,
          "com CAIXA ALTA em excesso": (_txt.str.count(r"[A-ZÁÉÍÓÚÂÊÔÃÕÇ]") > .3 * _txt.str.len()).to_numpy()}
linhas = []
for nome_g, m in grupos.items():
    if m.sum() >= 20 and len(set(y_va[m])) == 2:
        linhas.append({"grupo": nome_g, "textos": int(m.sum()),
                       "F1-macro": round(f1_score(y_va[m], previsoes[m], average="macro"), 3)})
display(pd.DataFrame(linhas))
print("A nota geral é a do placar. Os grupos de baixo é que fazem um bom relatório: onde o")
print("modelo cai, e por quê. (Grupos com menos de 20 textos ou uma classe só ficam de fora.)")

In [ ]:
#@title 🧪 Testem com uma frase de vocês { display-mode: "form" }
sua_frase = "Parabéns pela agilidade, só levou um mês pra chegar."  #@param {type:"string"}
_e = tradutor([sua_frase], truncation=True, padding="max_length", max_length=limite, return_tensors="pt")
with torch.no_grad():
    _p = torch.softmax(modelo(input_ids=_e["input_ids"].to(DISPOSITIVO),
                              attention_mask=_e["attention_mask"].to(DISPOSITIVO)).logits, 1)[0].cpu().numpy()
print(f'"{sua_frase}"\n')
print(f"   → {NOME_CLASSE[int(_p.argmax())]}   ({_p.max()*100:.0f}% segundo a conta interna do modelo)")
print("\n   Testem ironia, negação ('não é ruim') e um elogio ao produto com reclamação do prazo.")

In [ ]:
#@title ⏱️ Custo contra ganho: os experimentos desta sessão { display-mode: "form" }
display(pd.DataFrame(registro))
print("O DistilBERT é ~2× mais rápido; o XLM-R, o mais pesado. Anotem quanto cada ponto")
print("de F1 custou em minutos de GPU: é essa tabela que vira o relatório.")

---
# 🤔 PARADA OBRIGATÓRIA: o modelo que vocês usaram não é de vocês

1. **Ele herda o que estava torto.** O BERTimbau aprendeu português com textos da web; o
   XLM-R, com cem idiomas. Que tipo de escrita está sub-representada nesse material? Os
   grupos de texto do Bloco 4 mostraram algum indício?
2. **O rótulo é a estrela.** O modelo aprende a relação texto → estrela, não a satisfação.
   Ironia e reclamação de prazo com elogio ao produto continuam difíceis — por quê?
3. **O mesmo modelo, dois usos.** Priorizar o atendimento ou esconder avaliações negativas
   na página do produto: qual restrição de uso vocês anexariam?
4. **IA generativa.** Podem usar, desde que saibam explicar o que ela entregou: registrem
   qual, para quê, como conferiram e o que mudaram.

In [ ]:
#@title 📝 O relatório da sua equipe { display-mode: "form" }
nome_da_equipe = ""  #@param {type:"string"}
#@markdown **1. Qual modelo e quais valores dos quatro botões? Por que esse limite (use o número do Bloco 1)?**
resposta_1 = ""  #@param {type:"string"}
#@markdown **2. O que as curvas mostraram? Em que época estava o melhor ponto?**
resposta_2 = ""  #@param {type:"string"}
#@markdown **3. Em que grupo de texto o modelo cai? Quantos FN e FP, e qual custa mais aqui?**
resposta_3 = ""  #@param {type:"string"}
#@markdown **4. O modelo é emprestado e o rótulo vem da estrela: que limites isso impõe, e que restrição de uso vocês anexariam?**
resposta_4 = ""  #@param {type:"string"}
#@markdown **5. Usaram IA generativa? Qual, para quê, como conferiram e o que mudaram?**
resposta_5 = ""  #@param {type:"string"}
relatorio = {"equipe": nome_da_equipe, "modelo": modelo_escolhido, "nome_hf": NOME, "taxa": taxa,
             "epocas": epocas, "melhor_epoca": historico["melhor_epoca"], "limite": limite,
             "aquecimento": aquecimento, "lote": lote, "fracao_do_acervo": fracao_do_acervo,
             "piso": round(piso, 4), "f1_macro": round(f1, 4), "acuracia": round(acuracia, 4),
             "recall_critica": round(recall_critica, 4), "precisao_critica": round(precisao_critica, 4),
             "just_configuracao": resposta_1, "just_curvas": resposta_2, "just_grupos_e_erros": resposta_3,
             "just_etica": resposta_4, "uso_ia_generativa": resposta_5}
faltando = [k for k in ["equipe", "just_configuracao", "just_curvas", "just_grupos_e_erros",
                        "just_etica", "uso_ia_generativa"] if not str(relatorio[k]).strip()]
print("⚠️ Ainda faltam campos: " + ", ".join(faltando) if faltando
      else "Relatório completo. Prossigam para a submissão.")

In [ ]:
#@title 🚀 Gerar o arquivo de submissão { display-mode: "form" }
import json, pathlib
PASTA_SUBMISSAO = pathlib.Path("submissoes")
PASTA_SUBMISSAO.mkdir(parents=True, exist_ok=True)
if "relatorio" not in dir() or not isinstance(relatorio, dict):
    relatorio = {"equipe": "", "modelo": modelo_escolhido, "f1_macro": round(f1, 4)}
# O teste passa pelo MESMO tradutor, com o MESMO limite: texto cru, sem limpeza.
modelo.eval()
palpites = prever(modelo, carregador(tokenizar(mensagens_teste), np.zeros(N_TESTE, int), 64, False)).argmax(axis=1)
submissao = pd.DataFrame({"id_mensagem": IDS_TESTE, "rotulo_previsto": palpites})

assert len(submissao) == N_TESTE, f"A submissão precisa ter {N_TESTE} linhas."
csv_saida = PASTA_SUBMISSAO / "submissao_voz_do_cliente.csv"
json_saida = PASTA_SUBMISSAO / "relatorio_voz_do_cliente.json"
submissao.to_csv(csv_saida, index=False)
with open(json_saida, "w", encoding="utf-8") as f:
    json.dump(relatorio, f, ensure_ascii=False, indent=2)

print("="*60)
print("             ARQUIVOS PRONTOS PARA ENVIO")
print("="*60)
print(f"  submissao_voz_do_cliente.csv   ({len(submissao)} linhas)")
print("  relatorio_voz_do_cliente.json")
print("="*60)
_n0 = int((palpites == CLASSE_CRITICA).sum())
print(f"\n  Marcados como {NOME_CLASSE[CLASSE_CRITICA].lower()}: {_n0} de {N_TESTE} ({_n0/N_TESTE*100:.1f}%). "
      f"No acervo: {(dados.rotulo == 0).mean()*100:.1f}%.")
display(submissao.head())
print("\n  Para corrigir (da raiz do repositório):")
print(f"     python simulados/corrigir.py voz_do_cliente {csv_saida}")

try:
    from google.colab import files
    files.download(str(csv_saida))
    files.download(str(json_saida))
    print("\n  Download iniciado.")
except ImportError:
    print(f"\n  Fora do Colab: os arquivos estão em {PASTA_SUBMISSAO.resolve()}")

---
# 🌟 FIM DA OPERAÇÃO VOZ DO CLIENTE · FASE 2

### Competências praticadas

- Pedaços de palavra: por que palavra nunca vista não some mais
- A regra do mesmo nome para tradutor e modelo
- Texto cru: por que não limpar como na Fase 1
- O limite de pedaços medido no corpus, e o custo de movê-lo
- Taxa, épocas, aquecimento e a leitura das curvas de ajuste fino
- Matriz com nome próprio, recall como apoio, F1 por grupo de texto
- Comparação honesta com a marca da Fase 1